# 03 - Temporal Drift Analysis

Detect distribution shifts across quarters using Kolmogorov-Smirnov and chi-squared tests. Identify when and where data distributions change significantly.

**Toolkit modules used**: `cj_data_quality.drift`, `cj_data_quality.visualization`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from cj_data_quality.notebook_utils import setup_notebook
from cj_data_quality.sample_data import generate_and_load
from cj_data_quality.drift.distribution_drift import (
    detect_numeric_drift,
    detect_categorical_drift,
    classify_drift_severity,
)
from cj_data_quality.drift.temporal_drift import detect_temporal_drift, summarize_drift_over_time
from cj_data_quality.visualization.plots import plot_drift_timeline

setup_notebook()
print("Setup complete.")

## Load Data

In [ ]:
df = generate_and_load(n_records=50000, seed=42)
print(f"Shape: {df.shape}")
print(f"Date range: {df['reporting_date'].min()} to {df['reporting_date'].max()}")

## Numeric Drift: Population Counts Across Quarters

Compare total_population distributions between consecutive quarters using the KS test.

In [ ]:
# Filter to a single state for cleaner analysis
ca_df = df[df["state_code"] == "US_CA"].dropna(subset=["reporting_date", "total_population"]).copy()
ca_df["quarter"] = ca_df["reporting_date"].dt.to_period("Q")

quarters = sorted(ca_df["quarter"].unique())
print(f"Quarters available: {len(quarters)}")

drift_results = []
for i in range(len(quarters) - 1):
    ref = ca_df[ca_df["quarter"] == quarters[i]]["total_population"]
    comp = ca_df[ca_df["quarter"] == quarters[i + 1]]["total_population"]
    if len(ref) > 5 and len(comp) > 5:
        result = detect_numeric_drift(
            ref, comp, "total_population",
            str(quarters[i]), str(quarters[i + 1])
        )
        drift_results.append(result)

print(f"\nDrift tests run: {len(drift_results)}")
for r in drift_results[:5]:
    print(f"  {r.reference_period} → {r.comparison_period}: p={r.p_value:.4f} ({r.severity.value})")

## Drift Timeline Visualization

In [ ]:
if drift_results:
    drift_df = pd.DataFrame([
        {"period_pair": f"{r.reference_period}→{r.comparison_period}", "p_value": r.p_value}
        for r in drift_results
    ])
    fig = plot_drift_timeline(drift_df, title="California Population Distribution Drift")
    plt.show()

## Categorical Drift: Race Distribution Changes

In [ ]:
ca_race = ca_df.dropna(subset=["race"])

if len(quarters) >= 2:
    first_q = ca_race[ca_race["quarter"] == quarters[0]]["race"]
    last_q = ca_race[ca_race["quarter"] == quarters[-1]]["race"]
    
    if len(first_q) > 5 and len(last_q) > 5:
        cat_result = detect_categorical_drift(
            first_q, last_q, "race",
            str(quarters[0]), str(quarters[-1])
        )
        print(f"Race distribution drift ({quarters[0]} → {quarters[-1]}):")
        print(f"  Chi-squared statistic: {cat_result.statistic:.4f}")
        print(f"  p-value: {cat_result.p_value:.4f}")
        print(f"  Severity: {cat_result.severity.value}")

## Multi-State Temporal Drift

Run drift detection across all states simultaneously.

In [ ]:
subset = df[df["state_code"].isin(["US_CA", "US_TX", "US_NY", "US_FL", "US_OH"])].copy()
subset = subset.dropna(subset=["reporting_date", "total_population"])

temporal_results = detect_temporal_drift(
    subset, date_col="reporting_date", value_col="total_population",
    period="Q", group_col="state_code"
)

print(f"Total drift results: {len(temporal_results)}")
significant = [r for r in temporal_results if r.severity.value != "none"]
print(f"Significant drift events: {len(significant)}")

for r in significant[:10]:
    print(f"  {r.column_name} {r.reference_period}→{r.comparison_period}: {r.severity.value} (p={r.p_value:.6f})")

## Drift Summary Table

In [ ]:
if temporal_results:
    summary = summarize_drift_over_time(temporal_results)
    summary.head(20)

## Key Findings

1. **Population drift** varies by state — injected spikes create detectable distribution shifts
2. **Categorical drift** in race distributions is generally low (expected for synthetic data)
3. **The KS test** is sensitive enough to detect the injected population anomalies
4. **Temporal drift analysis** across quarters provides an early warning system for data pipeline issues

In production, this would catch scenarios like a state changing its reporting methodology mid-year.